In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import LearningRateScheduler, EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import shutil

# Set image dimensions
IMG_HEIGHT = 256
IMG_WIDTH = 256

# Convert YOLO label format to binary mask
def yolo_to_mask(label_file, img_shape):
    mask = np.zeros((img_shape[0], img_shape[1]), dtype=np.uint8)
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f.readlines():
                class_id, x_center, y_center, width, height = map(float, line.strip().split())
                x_center *= img_shape[1]
                y_center *= img_shape[0]
                width *= img_shape[1]
                height *= img_shape[0]
                x_min = int(x_center - width / 2)
                y_min = int(y_center - height / 2)
                x_max = int(x_center + width / 2)
                y_max = int(y_center + height / 2)
                mask[y_min:y_max, x_min:x_max] = 1
    return mask

# Load dataset with YOLO labels
def load_yolo_dataset(image_dir, label_dir):
    images, masks = [], []
    for filename in os.listdir(image_dir):
        if filename.endswith(('.jpg', '.png')):
            img_path = os.path.join(image_dir, filename)
            img = cv2.imread(img_path)
            img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
            img = img / 255.0
            images.append(img)
            
            label_filename = filename.replace('.jpg', '.txt').replace('.png', '.txt')
            label_path = os.path.join(label_dir, label_filename)
            mask = yolo_to_mask(label_path, (IMG_HEIGHT, IMG_WIDTH))
            masks.append(mask)
    images = np.array(images)
    masks = np.array(masks)
    masks = np.expand_dims(masks, axis=-1)
    return images, masks

# Data generator
def data_generator(image_dir, label_dir, batch_size):
    image_filenames = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png'))]
    while True:
        np.random.shuffle(image_filenames)
        for start in range(0, len(image_filenames), batch_size):
            end = min(start + batch_size, len(image_filenames))
            batch_filenames = image_filenames[start:end]
            batch_images, batch_masks = [], []
            for filename in batch_filenames:
                img_path = os.path.join(image_dir, filename)
                img = cv2.imread(img_path)
                img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
                img = img / 255.0
                batch_images.append(img)
                
                label_filename = filename.replace('.jpg', '.txt').replace('.png', '.txt')
                label_path = os.path.join(label_dir, label_filename)
                mask = yolo_to_mask(label_path, (IMG_HEIGHT, IMG_WIDTH))
                batch_masks.append(mask)
            yield np.array(batch_images), np.expand_dims(np.array(batch_masks), axis=-1)

# Dice and Binary Cross-Entropy combined loss
def dice_bce_loss(y_true, y_pred):
    bce = tf.keras.losses.BinaryCrossentropy()
    bce_loss = bce(y_true, y_pred)
    smooth = 1.0
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    dice_loss = 1 - (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)
    return 0.5 * bce_loss + 0.5 * dice_loss





# Learning rate schedule
def lr_schedule(epoch, lr):
    if epoch < 10:
        return lr
    elif 10 <= epoch < 20:
        return lr * 0.1
    else:
        return lr * 0.01

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

# Main execution
if __name__ == "__main__":
    train_image_dir = '/kaggle/input/dataset-lunar-crater/LU3M6TGT_yolo_format/train/images'
    train_label_dir = '/kaggle/input/dataset-lunar-crater/LU3M6TGT_yolo_format/train/labels'
    valid_image_dir = '/kaggle/input/dataset-lunar-crater/LU3M6TGT_yolo_format/valid/images'
    valid_label_dir = '/kaggle/input/dataset-lunar-crater/LU3M6TGT_yolo_format/valid/labels'

    epochs = 25
    batch_size = 32
    input_shape = (IMG_HEIGHT, IMG_WIDTH, 3)
    model = create_deeper_resnet_unet(input_shape)

    initial_lr = 1e-3
    model.compile(optimizer=AdamW(learning_rate=initial_lr),
                  loss=dice_bce_loss,
                  metrics=['accuracy'])

    train_gen = data_generator(train_image_dir, train_label_dir, batch_size)
    val_gen = data_generator(valid_image_dir, valid_label_dir, batch_size)

    num_train_samples = len(os.listdir(train_image_dir))
    num_val_samples = len(os.listdir(valid_image_dir))
    steps_per_epoch = num_train_samples // batch_size
    validation_steps = num_val_samples // batch_size

    lr_scheduler = LearningRateScheduler(lr_schedule, verbose=1)

    # Define where to save the model
    model_save_path = "resnet_unet_model.keras"

    # Model checkpoint callback to save the best model
    model_checkpoint = ModelCheckpoint(
        filepath=model_save_path,
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )

    # Include the callbacks
    callbacks = [lr_scheduler, early_stopping, model_checkpoint]

    # Training the model
    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_gen,
        validation_steps=validation_steps,
        epochs=epochs,
        callbacks=callbacks
    )

    # Compress and download the model
    shutil.make_archive('resnet_unet_model', 'zip', '.', 'resnet_unet_model.keras')
    from IPython.display import FileLink
    display(FileLink('resnet_unet_model.zip'))


Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/25


I0000 00:00:1732091069.010989      95 service.cc:145] XLA service 0x7fc814005b60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1732091069.011050      95 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1732091116.793359      95 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'loop_add_subtract_fusion_25', 20 bytes spill stores, 20 bytes spill loads

I0000 00:00:1732091116.825307      95 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 990ms/step - accuracy: 0.8477 - loss: 0.4352
Epoch 1: val_loss improved from inf to 0.48100, saving model to resnet_unet_model.keras
269/269 ━━━━━━━━━━━━━━━━━━━━ 373s 1s/step - accuracy: 0.8478 - loss: 0.4350 - val_accuracy: 0.8687 - val_loss: 0.4810 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 2/25
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 992ms/step - accuracy: 0.9051 - loss: 0.2953
Epoch 2: val_loss did not improve from 0.48100
269/269 ━━━━━━━━━━━━━━━━━━━━ 334s 1s/step - accuracy: 0.9051 - loss: 0.2952 - val_accuracy: 0.7699 - val_loss: 0.5119 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 3/25
269/269 ━━━━━━━━━━━━━━━━━━━━ 0s 992ms/step - accuracy: 0.9188 - loss: 0.2566
Epoch 3: val_loss did not improve from 0.48100
269/269 ━━━━━━━━━━━━━━━━━━━━ 284s 1s/step - accuracy: 0.9188 - loss: 0.2565 - val_accuracy: 0.8828 - val_loss: 0.5366 - l

/kaggle/working/resnet_unet_model.zip

In [1]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

# Load the trained model
model = load_model("resnet_unet_model.h5", custom_objects={"dice_bce_loss": dice_bce_loss, "iou_metric": iou_metric})

# Helper function: Preprocess input image
def preprocess_image(image_path, target_size):
    # Read and resize the image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized_image = cv2.resize(image, target_size)  # Resize to model input size
    normalized_image = resized_image / 255.0       # Normalize to [0, 1]
    return image, np.expand_dims(normalized_image, axis=0)

# Helper function: Post-process mask and draw bounding boxes
def postprocess_and_draw_boxes(original_image, predicted_mask):
    # Threshold the mask to binary
    binary_mask = (predicted_mask > 0.5).astype(np.uint8)
    
    # Find contours in the binary mask
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Draw bounding boxes around detected contours
    output_image = original_image.copy()
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(output_image, (x, y), (x+w, y+h), (255, 0, 0), 2)  # Blue box with thickness 2
    
    return output_image

# Main function to predict and draw bounding boxes
def predict_and_visualize(image_path):
    # Step 1: Preprocess input image
    target_size = (256, 256)  # Replace with your model's input size
    original_image, preprocessed_image = preprocess_image(image_path, target_size)
    
    # Step 2: Predict the mask
    predicted_mask = model.predict(preprocessed_image)[0, :, :, 0]  # Remove batch and channel dimensions
    
    # Step 3: Resize predicted mask to original image size
    predicted_mask_resized = cv2.resize(predicted_mask, (original_image.shape[1], original_image.shape[0]))
    
    # Step 4: Post-process mask and draw bounding boxes
    output_image = postprocess_and_draw_boxes(original_image, predicted_mask_resized)
    
    # Step 5: Display input and output images
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title("Original Image")
    plt.imshow(original_image)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Output with Bounding Boxes")
    plt.imshow(output_image)
    plt.axis("off")

    plt.show()

# Example usage
image_path = "/path/to/your/input_image.jpg"  # Replace with the path to your input image
predict_and_visualize(image_path)

NameError: name 'dice_bce_loss' is not defined